# **Climate ETL - Single-Province Example**

This notebook documents the end-to-end transformation applied to one provincial climate dataset.

The source dataset is stored as a gzip-compressed CSV object in Google Cloud Storage and is read directly into pandas. The resulting monthly provincial dataset is exported to the local `data/` directory.

The notebook provides an executable example of the transformation logic later reused by the batch pipeline for all provincial source files.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Environment and data locations

The notebook uses a hybrid data layout:

- raw climate datasets are stored as gzip-compressed CSV objects in Google Cloud Storage;
- the notebook is executed locally;
- the generated example output is saved in the local project `data/` directory.

The source objects follow this structure:

```text
gs://agriclimate-intelligence-data/
└── raw/
    └── climate/
        └── v1/
            └── <province>.csv.gz
```
The relevant local project structure is:
```text
project/

├── data/

│   └── TEST_<province>.csv

└── notebooks/

    └── ETL_Climate_Example.ipynb
```
The local `data/` directory must already exist. Access to Google Cloud Storage requires valid Application Default Credentials, while gcsfs must be available in the active Python environment.

In [2]:
# Google Cloud resources containing the raw climate datasets
PROJECT_ID = "agriclimate-intelligence"
BUCKET_NAME = "agriclimate-intelligence-data"

In [3]:
# Resolve the local project root from the notebook location.
# The notebook is expected to be executed from the project's notebooks directory.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent

# GCS prefix containing the gzip-compressed source datasets
INPUT_DIR = f"gs://{BUCKET_NAME}/raw/climate/v1"

# Local directory used for the generated example output
OUTPUT_DIR = PROJECT_ROOT / "data"

# Options passed by pandas to gcsfs.
# Authentication is provided through Google Cloud Application Default Credentials.
GCS_STORAGE_OPTIONS = {
    "project": PROJECT_ID
}

## Source object selected for the example

This notebook processes one provincial dataset so that every transformation and validation step can be inspected individually.

`FILENAME` therefore identifies one gzip-compressed CSV object stored under the configured Google Cloud Storage prefix.

In the production batch pipeline, source filenames are obtained from the province mapping and processed iteratively. Some production areas may require multiple source objects to be read and concatenated before the provincial transformation is applied.

In [4]:
FILENAME = 'alessandria.csv.gz'

input_path = f"{INPUT_DIR}/{FILENAME}"

### Direct read from Google Cloud Storage

The selected object is opened by `pandas.read_csv()` through `gcsfs`.

The gzip-compressed data are transferred from Google Cloud Storage and decompressed during reading. The notebook does not create a persistent local copy of the original compressed file.

In [5]:
df = pd.read_csv(
    input_path, 
    sep = ';',
    compression="gzip",
    storage_options=GCS_STORAGE_OPTIONS,
    dtype = {"LATITUDE" : "string", "LONGITUDE" : "string"}
)

display(df.head())
display(df.tail())
df.info()

,IDCELL,LATITUDE,LONGITUDE,ALTITUDE,DAY,TEMPERATURE_MAX,TEMPERATURE_MIN,TEMPERATURE_AVG,WINDSPEED,VAPOURPRESSURE,PRECIPITATION,ET0,RADIATION
0,420511,44.6541,8.4144,174,1980-01-01 00:00:00,7.7,-3.4,2.2,1.7,4.0,1,0.6,6393
1,420511,44.6541,8.4144,174,1980-01-02 00:00:00,7.4,-5.5,1.0,3.1,3.7,2,1.0,6435
2,420511,44.6541,8.4144,174,1980-01-03 00:00:00,6.7,-3.2,1.8,2.8,3.7,0,0.9,6643
3,420511,44.6541,8.4144,174,1980-01-04 00:00:00,3.3,-4.4,-0.6,0.4,4.6,0,0.2,5130
4,420511,44.6541,8.4144,174,1980-01-05 00:00:00,1.1,-2.7,-0.8,0.0,5.7,5,0.2,2525


,IDCELL,LATITUDE,LONGITUDE,ALTITUDE,DAY,TEMPERATURE_MAX,TEMPERATURE_MIN,TEMPERATURE_AVG,WINDSPEED,VAPOURPRESSURE,PRECIPITATION,ET0,RADIATION
940907,425509,44.84205,9.04046,311,2025-12-27 00:00:00,9.3,-0.3,4.5,0.9,6.4,0,0.4,5748
940908,425509,44.84205,9.04046,311,2025-12-28 00:00:00,12.7,4.8,8.8,1.3,5.7,0,0.7,5744
940909,425509,44.84205,9.04046,311,2025-12-29 00:00:00,11.4,1.6,6.5,1.1,5.7,0,0.5,6012
940910,425509,44.84205,9.04046,311,2025-12-30 00:00:00,9.3,4.7,7.0,0.4,8.4,0,0.5,4291
940911,425509,44.84205,9.04046,311,2025-12-31 00:00:00,6.1,0.7,3.4,0.3,6.7,0,0.3,3999


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 940912 entries, 0 to 940911
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   IDCELL           940912 non-null  int64  
 1   LATITUDE         940912 non-null  string 
 2   LONGITUDE        940912 non-null  string 
 3   ALTITUDE         940912 non-null  int64  
 4   DAY              940912 non-null  object 
 5   TEMPERATURE_MAX  940912 non-null  float64
 6   TEMPERATURE_MIN  940912 non-null  float64
 7   TEMPERATURE_AVG  940912 non-null  float64
 8   WINDSPEED        940912 non-null  float64
 9   VAPOURPRESSURE   940912 non-null  float64
 10  PRECIPITATION    940912 non-null  int64  
 11  ET0              940912 non-null  float64
 12  RADIATION        940912 non-null  int64  
dtypes: float64(6), int64(4), object(1), string(2)
memory usage: 93.3+ MB


In [6]:
# Forcing the date field as a datetime type
df["DAY"] = pd.to_datetime(df["DAY"], errors="raise" ).dt.normalize()

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 940912 entries, 0 to 940911
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   IDCELL           940912 non-null  int64         
 1   LATITUDE         940912 non-null  string        
 2   LONGITUDE        940912 non-null  string        
 3   ALTITUDE         940912 non-null  int64         
 4   DAY              940912 non-null  datetime64[ns]
 5   TEMPERATURE_MAX  940912 non-null  float64       
 6   TEMPERATURE_MIN  940912 non-null  float64       
 7   TEMPERATURE_AVG  940912 non-null  float64       
 8   WINDSPEED        940912 non-null  float64       
 9   VAPOURPRESSURE   940912 non-null  float64       
 10  PRECIPITATION    940912 non-null  int64         
 11  ET0              940912 non-null  float64       
 12  RADIATION        940912 non-null  int64         
dtypes: datetime64[ns](1), float64(6), int64(4), string(2)
memory usage: 93.3 M

In [8]:
# Creation of a few features derived from the date
df["YEAR"] = df["DAY"].dt.year
df["MONTH"] = df["DAY"].dt.month
df["YEAR_MONTH"] = df["DAY"].dt.to_period("M")
df["DAY_OF_YEAR"] = df["DAY"].dt.dayofyear

In [9]:
print('Number of null values for each column')
display(df.isna().sum())

Number of null values for each column


IDCELL             0
LATITUDE           0
LONGITUDE          0
ALTITUDE           0
DAY                0
TEMPERATURE_MAX    0
TEMPERATURE_MIN    0
TEMPERATURE_AVG    0
WINDSPEED          0
VAPOURPRESSURE     0
PRECIPITATION      0
ET0                0
RADIATION          0
YEAR               0
MONTH              0
YEAR_MONTH         0
DAY_OF_YEAR        0
dtype: int64

## Basic quality checks

The following checks are displayed directly because this notebook processes a single source file.

In the batch pipeline, the validation results can instead be collected in a structured report, while blocking issues can stop the transformation by raising an exception.

In [10]:
print('Duplicated cells:',int(df.duplicated(subset=["IDCELL", "DAY"], keep = False).sum()))

Duplicated cells: 0


Here below is performed a simple integrity check on the cell IDs: each ID should have just one possible value of latitude, longitude and altitude, otherwise it means that the grid definition has changed over time (in that case it would be necessary to see in detail what happened and when).

In [11]:
display(df.groupby("IDCELL").agg(
              latitude_values=("LATITUDE", "nunique"),
              longitude_values=("LONGITUDE", "nunique"),
              altitude_values=("ALTITUDE", "nunique")
          )
       )

,latitude_values,longitude_values,altitude_values
IDCELL,,,
418505,1,1,1
418506,1,1,1
418507,1,1,1
419505,1,1,1
419506,1,1,1
419507,1,1,1
419511,1,1,1
419512,1,1,1
419513,1,1,1


In [12]:
# Number of different IDs
display(df['IDCELL'].nunique())

56

The number of observations is calculated for each cell ID.

If every cell ID has the same observation count, the resulting frequency distribution contains a single value. This indicates that the set of grid cells remained unchanged throughout the observed period.

In [13]:
if df['IDCELL'].value_counts().value_counts().shape[0] != 1:
    print('There has been a change in the cells during the time')
    display(df['IDCELL'].value_counts())
else:
    print('All cells have been consistent during the time')

All cells have been consistent during the time


A simple control on temperatures, the minimum cannot be bigger than the average or the maximum, and the average cannot be bigger than the maximum.

In [14]:
invalid_temperatures = ((df["TEMPERATURE_MIN"] > df["TEMPERATURE_AVG"]) 
                        | (df["TEMPERATURE_AVG"] > df["TEMPERATURE_MAX"])
                        | (df["TEMPERATURE_MIN"] > df["TEMPERATURE_MAX"])
    )

In [15]:
if invalid_temperatures.any():
    print('There are invalid temperature values in the dataset')
else:
    print('Temperature data are OK')

Temperature data are OK


The following variables are expected to contain only non-negative values.

In [16]:
not_neg = True
for column in ["WINDSPEED", "PRECIPITATION", "ET0", "RADIATION"]:
    if (df[column] < 0).any():
        print("There are negative values in ",column)
        not_neg = False
if not_neg:
    print("There are no negative values for wind speed, precipitation, ET0 or radiation")

There are no negative values for wind speed, precipitation, ET0 or radiation


### Weight of geographic data point estimation

Since all cells of the grids are equal in size, the weight carried by each one is 1 / number of cells in a given province.

(Here it is assumed that the previous checks on the cell IDs showed that the same IDs repeated each day available in the dataset)

In [17]:
AREA_WEIGHT = 1 / df['IDCELL'].nunique()

### Definition of wet days
It is estimated that a wet day for a single cell is when there is at least 1 mm of rain

In [18]:
df["WET_CELL"] = df["PRECIPITATION"] >= 1

## Threshold estimation

Relative-event thresholds are estimated separately for each province and calendar month using the 1980–2010 reference period.

The current configuration includes:

- 90th percentile of maximum temperature;
- 95th percentile of maximum temperature;
- 10th percentile of minimum temperature;
- 95th percentile of wind speed;
- 95th percentile of positive precipitation values;
- 99th percentile of positive precipitation values.

The 1980–2010 period acts as the climatic reference baseline against which subsequent observations are compared. The threshold configuration can be revised if a more appropriate official or domain-specific definition is adopted.

Those thresholds are computed on the first 30 years in the dataset, which will be the reference for what is considered a typical metereological situation.

In [19]:
pre_2011 = df.loc[df["YEAR"].lt(2011)]

In [20]:
temperature_thresholds = (
        pre_2011.groupby("MONTH")
          .agg(
              TMAX_P90=(
                  "TEMPERATURE_MAX",
                  lambda values: values.quantile(0.90)
              ),
              TMAX_P95=(
                  "TEMPERATURE_MAX",
                  lambda values: values.quantile(0.95)
              ),
              TMIN_P10=(
                  "TEMPERATURE_MIN",
                  lambda values: values.quantile(0.10)
              )
          )
          .reset_index()
    )

In [21]:
wind_thresholds = (
        pre_2011.groupby("MONTH")
          .agg(
              WIND_P95=(
                  "WINDSPEED",
                  lambda values: values.quantile(0.95)
              )
          )
          .reset_index()
    )

In [22]:
precipitation_thresholds = (
        pre_2011.loc[pre_2011["PRECIPITATION"] > 0].groupby("MONTH")
                .agg(
                    PRECIP_P95=(
                        "PRECIPITATION",
                        lambda values: values.quantile(0.95)
                    ),
                    PRECIP_P99=(
                        "PRECIPITATION",
                        lambda values: values.quantile(0.99)
                    ),
                )
                .reset_index()
    )

In [23]:
thresholds = temperature_thresholds.merge(
        precipitation_thresholds,
        on="MONTH",
        how="left",
        validate="one_to_one"
    )

thresholds = thresholds.merge(
        wind_thresholds,
        on="MONTH",
        how="left",
        validate="one_to_one"
    )

In [24]:
display(thresholds)

,MONTH,TMAX_P90,TMAX_P95,TMIN_P10,PRECIP_P95,PRECIP_P99,WIND_P95
0,1,10.9,12.4,-6.3,20.0,35.0,3.7
1,2,14.3,15.8,-5.0,22.0,41.0,3.8
2,3,19.6,21.0,-1.7,25.0,43.0,4.3
3,4,22.7,24.2,2.0,25.0,40.0,4.5
4,5,27.3,28.8,6.9,23.0,35.0,4.1
5,6,31.4,33.0,10.2,18.0,32.0,3.8
6,7,33.2,34.3,12.9,16.0,29.0,3.3
7,8,32.8,34.0,12.9,21.0,37.0,3.5
8,9,28.3,29.5,8.3,28.0,59.0,3.9
9,10,22.6,24.0,3.5,33.0,55.0,4.2


### Creation of cell-level indicators

In this section there is the definition of two kind of indicators, defined as follows:
- values comparison to a fixed, pre-determined threshold
- values comparison to the previously defined thresholds, obtained from the quantiles of the variables, observed from 1980 to 2010

In general, here are computed indicators that highlight the presence or absence of a particularly relevant phenomenon in a given space unit, with the goal later on of checking if it happened in a relevant part of the province.

In [25]:
# Add the monthly threshold columns to each cell-day record for direct comparison
df = df.merge(
        thresholds,
        on="MONTH",
        how="left",
        validate="many_to_one"
    )

Definition with fixed thresholds

In [26]:
df["FROST"] = df["TEMPERATURE_MIN"] < 0
df["HOT_30"] = df["TEMPERATURE_MAX"] >= 30
df["HOT_35"] = df["TEMPERATURE_MAX"] >= 35
df["RAIN_20"] = df["PRECIPITATION"] >= 20
df["RAIN_50"] = df["PRECIPITATION"] >= 50

Definition with the computed thresholds

In [27]:
df["EXTREME_HEAT"] = (df["TEMPERATURE_MAX"] > df["TMAX_P90"])

df["VERY_EXTREME_HEAT"] = (df["TEMPERATURE_MAX"] > df["TMAX_P95"])

df["EXTREME_COLD"] = (df["TEMPERATURE_MIN"] < df["TMIN_P10"])

df["EXTREME_RAIN"] = ((df["PRECIPITATION"] >= 1) & (df["PRECIPITATION"] > df["PRECIP_P95"]))

df["VERY_EXTREME_RAIN"] = ((df["PRECIPITATION"] >= 1) & (df["PRECIPITATION"] > df["PRECIP_P99"]))

df["EXTREME_WIND"] = (df["WINDSPEED"] > df["WIND_P95"])

### Function for temporary daily aggregation of data

In [28]:
flag_columns = [
        "FROST",
        "HOT_30",
        "HOT_35",
        "RAIN_20",
        "RAIN_50",
        "EXTREME_HEAT",
        "VERY_EXTREME_HEAT",
        "EXTREME_COLD",
        "EXTREME_RAIN",
        "VERY_EXTREME_RAIN",
        "EXTREME_WIND",
        "WET_CELL"
    ]

In [29]:
# Store one dictionary per day; each dictionary becomes one output row.
# Aggregate numerical variables across all grid cells for each province-day.
# Reuse the grouped object throughout the loop.

def aggregate_to_daily_province(df, area_weight, flag_columns):
    
    # This will be a list of dictionaries, each representing a row of the dataframe which will be returned
    rows = []

    expected_cells = df["IDCELL"].nunique()

    for day, group in df.groupby(
        "DAY",
        sort=True,
        observed=True
    ):
        n_observed_cells = group["IDCELL"].nunique()

        # This safeguard should never trigger after the previous validation steps.
        # Build one dictionary containing the aggregated values for the current day.
        if n_observed_cells != expected_cells:
            raise ValueError(
                f"Il giorno {day:%Y-%m-%d} contiene "
                f"{n_observed_cells} celle, "
                f"mentre erano attese {expected_cells} celle."
            )

        # A single row is defined with a dictionary, in which values of a single day are grouped
        row = {
            "DAY": day,
            "N_CELLS": n_observed_cells,

            # Provincial spatial means
            "TEMPERATURE_AVG_AREA": (
                group["TEMPERATURE_AVG"].mean()
            ),
            "TEMPERATURE_MAX_AREA": (
                group["TEMPERATURE_MAX"].mean()
            ),
            "TEMPERATURE_MIN_AREA": (
                group["TEMPERATURE_MIN"].mean()
            ),

            # Local temperature extremes
            "TEMPERATURE_MAX_LOCAL": (
                group["TEMPERATURE_MAX"].max()
            ),
            "TEMPERATURE_MIN_LOCAL": (
                group["TEMPERATURE_MIN"].min()
            ),

            # Spatial temperature variability
            "TEMPERATURE_SPATIAL_STD": (
                group["TEMPERATURE_AVG"].std()
            ),

            # Provincial mean precipitation
            "PRECIPITATION_AREA": (
                group["PRECIPITATION"].mean()
            ),

            # Highest local precipitation
            "PRECIPITATION_LOCAL_MAX": (
                group["PRECIPITATION"].max()
            ),

            # Spatial precipitation variability
            "PRECIPITATION_SPATIAL_STD": (
                group["PRECIPITATION"].std()
            ),

            # Wind
            "WINDSPEED_AREA": (
                group["WINDSPEED"].mean()
            ),
            "WINDSPEED_LOCAL_MAX": (
                group["WINDSPEED"].max()
            ),

            # Other environmental variables
            "VAPOURPRESSURE_AREA": (
                group["VAPOURPRESSURE"].mean()
            ),
            "ET0_AREA": (
                group["ET0"].mean()
            ),
            "RADIATION_AREA": (
                group["RADIATION"].mean()
            ),
        }

        # For each boolean cell-level indicator, calculate the affected provincial area fraction.
        # Area-fraction columns use the suffix _AREA_FRACTION.
        for flag in flag_columns:
            affected_cells = int(group[flag].sum())

            row[f"{flag}_AREA_FRACTION"] = (
                affected_cells * area_weight
            )

        rows.append(row)

    daily = pd.DataFrame(rows)

    daily["YEAR"] = daily["DAY"].dt.year
    daily["MONTH"] = daily["DAY"].dt.month
    daily["YEAR_MONTH"] = (
        daily["DAY"].dt.to_period("M")
    )

    return daily

In [30]:
daily = aggregate_to_daily_province(
    df=df,
    area_weight=AREA_WEIGHT,
    flag_columns=flag_columns
)

In [31]:
display(daily.head())

,DAY,N_CELLS,TEMPERATURE_AVG_AREA,TEMPERATURE_MAX_AREA,TEMPERATURE_MIN_AREA,TEMPERATURE_MAX_LOCAL,TEMPERATURE_MIN_LOCAL,TEMPERATURE_SPATIAL_STD,PRECIPITATION_AREA,PRECIPITATION_LOCAL_MAX,...,EXTREME_HEAT_AREA_FRACTION,VERY_EXTREME_HEAT_AREA_FRACTION,EXTREME_COLD_AREA_FRACTION,EXTREME_RAIN_AREA_FRACTION,VERY_EXTREME_RAIN_AREA_FRACTION,EXTREME_WIND_AREA_FRACTION,WET_CELL_AREA_FRACTION,YEAR,MONTH,YEAR_MONTH
0,1980-01-01,56,2.010714,7.341071,-3.301786,9.4,-5.1,1.031094,0.857143,2,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.839286,1980,1,1980-01
1,1980-01-02,56,0.817857,6.973214,-5.337500,8.7,-8.8,1.136519,1.785714,3,...,0.0,0.0,0.053571,0.0,0.0,0.017857,0.982143,1980,1,1980-01
2,1980-01-03,56,1.510714,6.271429,-3.250000,7.8,-5.0,1.091734,0.000000,0,...,0.0,0.0,0.000000,0.0,0.0,0.107143,0.000000,1980,1,1980-01
3,1980-01-04,56,-0.660714,3.121429,-4.457143,5.7,-8.1,1.032504,0.107143,1,...,0.0,0.0,0.035714,0.0,0.0,0.000000,0.107143,1980,1,1980-01
4,1980-01-05,56,-1.046429,0.825000,-2.926786,6.0,-7.7,1.237882,5.428571,9,...,0.0,0.0,0.035714,0.0,0.0,0.000000,1.000000,1980,1,1980-01


For each cell-level boolean indicator, the daily provincial area fraction is compared with a minimum threshold.

An event is classified as province-relevant only when it affects at least that share of the provincial grid. Events below the threshold are treated as localized phenomena.

In [32]:
def add_daily_province_flags(daily, minimum_area_fraction=0.10):
    daily = daily.copy()

    # Relative precipitation extremes
    daily["EXTREME_RAIN_DAY"] = (
        daily["EXTREME_RAIN_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    daily["VERY_EXTREME_RAIN_DAY"] = (
        daily["VERY_EXTREME_RAIN_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    # Relative temperature extremes
    daily["EXTREME_HEAT_DAY"] = (
        daily["EXTREME_HEAT_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    daily["VERY_EXTREME_HEAT_DAY"] = (
        daily["VERY_EXTREME_HEAT_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    daily["EXTREME_COLD_DAY"] = (
        daily["EXTREME_COLD_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    # Wind extremes
    daily["EXTREME_WIND_DAY"] = (
        daily["EXTREME_WIND_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    # Absolute precipitation thresholds
    daily["RAIN_20_DAY"] = (
        daily["RAIN_20_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    daily["RAIN_50_DAY"] = (
        daily["RAIN_50_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    # Mean provincial precipitation >= 1 mm
    daily["WET_DAY_AREA_MEAN"] = (
        daily["PRECIPITATION_AREA"] >= 1
    )

    # At least the chosen fraction of the province received >= 1 mm
    daily["WET_DAY_SPATIAL"] = (
        daily["WET_CELL_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    # Provincial mean precipitation below 1 mm
    daily["DRY_DAY_AREA_MEAN"] = (
        daily["PRECIPITATION_AREA"] < 1
    )

    # No cell received at least 1 mm
    daily["DRY_DAY_ALL_CELLS"] = (
        daily["WET_CELL_AREA_FRACTION"] == 0
    )

    # Absolute temperature thresholds affecting at least
    # the selected fraction of the province
    daily["FROST_DAY_10PCT"] = (
        daily["FROST_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    daily["HOT_30_DAY_10PCT"] = (
        daily["HOT_30_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    daily["HOT_35_DAY_10PCT"] = (
        daily["HOT_35_AREA_FRACTION"]
        >= minimum_area_fraction
    )

    return daily

The following parameter defines the minimum share of provincial area required for a phenomenon to be classified as province-relevant.

In [33]:
MINIMUM_AREA_FRACTION = 0.10

In [34]:
daily = add_daily_province_flags(
    daily=daily,
    minimum_area_fraction=MINIMUM_AREA_FRACTION
)

In [35]:
display(daily.head())

,DAY,N_CELLS,TEMPERATURE_AVG_AREA,TEMPERATURE_MAX_AREA,TEMPERATURE_MIN_AREA,TEMPERATURE_MAX_LOCAL,TEMPERATURE_MIN_LOCAL,TEMPERATURE_SPATIAL_STD,PRECIPITATION_AREA,PRECIPITATION_LOCAL_MAX,...,EXTREME_WIND_DAY,RAIN_20_DAY,RAIN_50_DAY,WET_DAY_AREA_MEAN,WET_DAY_SPATIAL,DRY_DAY_AREA_MEAN,DRY_DAY_ALL_CELLS,FROST_DAY_10PCT,HOT_30_DAY_10PCT,HOT_35_DAY_10PCT
0,1980-01-01,56,2.010714,7.341071,-3.301786,9.4,-5.1,1.031094,0.857143,2,...,False,False,False,False,True,True,False,True,False,False
1,1980-01-02,56,0.817857,6.973214,-5.337500,8.7,-8.8,1.136519,1.785714,3,...,False,False,False,True,True,False,False,True,False,False
2,1980-01-03,56,1.510714,6.271429,-3.250000,7.8,-5.0,1.091734,0.000000,0,...,True,False,False,False,False,True,True,True,False,False
3,1980-01-04,56,-0.660714,3.121429,-4.457143,5.7,-8.1,1.032504,0.107143,1,...,False,False,False,False,True,True,False,True,False,False
4,1980-01-05,56,-1.046429,0.825000,-2.926786,6.0,-7.7,1.237882,5.428571,9,...,False,False,False,True,True,False,False,True,False,False


### Monthly aggregation

The monthly dataset is built through two parallel aggregation paths:
- continuous monthly statistics are computed directly from all cell-day observations in the original dataframe;
- indicators that explicitly depend on the daily provincial state are computed from the temporary daily dataframe.

The two results are then merged using year and month as keys.

In [36]:
# Return the mean calculated only over strictly positive values
def mean_when_positive(values):
    positive_values = values[values > 0]

    if positive_values.empty:
        return 0.0

    return positive_values.mean()

For extreme events, duration is also relevant.

The following dictionary maps each sequence indicator to the daily provincial flag used to calculate its maximum monthly run of consecutive days.

In [37]:
# Configuration of the daily provincial flags for which
# the maximum monthly consecutive-day sequence will be calculated
STREAK_FLAGS = {
    # Relative temperature extremes
    "MAX_CONSECUTIVE_EXTREME_HEAT_DAYS": (
        "EXTREME_HEAT_DAY"
    ),
    "MAX_CONSECUTIVE_VERY_EXTREME_HEAT_DAYS": (
        "VERY_EXTREME_HEAT_DAY"
    ),
    "MAX_CONSECUTIVE_EXTREME_COLD_DAYS": (
        "EXTREME_COLD_DAY"
    ),

    # Relative precipitation extremes
    "MAX_CONSECUTIVE_EXTREME_RAIN_DAYS": (
        "EXTREME_RAIN_DAY"
    ),
    "MAX_CONSECUTIVE_VERY_EXTREME_RAIN_DAYS": (
        "VERY_EXTREME_RAIN_DAY"
    ),

    # Wind extremes
    "MAX_CONSECUTIVE_EXTREME_WIND_DAYS": (
        "EXTREME_WIND_DAY"
    ),

    # Persistence of generally dry or wet conditions
    "MAX_CONSECUTIVE_DRY_DAYS_AREA_MEAN": (
        "DRY_DAY_AREA_MEAN"
    ),
    "MAX_CONSECUTIVE_WET_DAYS_AREA_MEAN": (
        "WET_DAY_AREA_MEAN"
    ),

    # Persistence of extreme temperature conditions based on absolute thresholds
    "MAX_CONSECUTIVE_FROST_10PCT_DAYS": (
        "FROST_DAY_10PCT"
    ),
    "MAX_CONSECUTIVE_HOT_30_10PCT_DAYS": (
        "HOT_30_DAY_10PCT"
    ),
    "MAX_CONSECUTIVE_HOT_35_10PCT_DAYS": (
        "HOT_35_DAY_10PCT"
    )
}

Next, it is defined a function to estimate the longest sequence of days, in a given month and for a given type of event

In [38]:
# The function computes the maximum number of consecutive calendar days
# in which the selected daily provincial condition is True.
# Missing calendar dates interrupt the sequence

def longest_consecutive_event_days(group, flag_column):
    ordered = (
        group.loc[:, ["DAY", flag_column]]
        .sort_values("DAY")
        .reset_index(drop=True)
    )

    if ordered.empty:
        return 0

    # Missing values are conservatively interpreted as no event
    event = (
        ordered[flag_column]
        .fillna(False)
        .astype(bool)
    )

    # A sequence can continue only if the current record is exactly
    # one calendar day after the previous record
    follows_previous_day = (
        ordered["DAY"]
        .diff()
        .eq(pd.Timedelta(days=1))
    )

    previous_day_was_event = (
        event.shift(fill_value=False)
    )

    # A new sequence starts when
    # the current day is an event, and
    # the previous day was not an event, or the calendar date is not consecutive
    starts_new_sequence = (
        event
        & (
            ~previous_day_was_event
            | ~follows_previous_day
        )
    )

    sequence_id = starts_new_sequence.cumsum()

    sequence_lengths = (
        event.groupby(sequence_id)
        .sum()
    )

    return int(sequence_lengths.max())

The following function applies the consecutive-day calculation to every configured event and returns one row for each year-month combination.

In [39]:
def aggregate_streaks_to_monthly(daily, streak_flags):

    # This safeguard should never trigger after the previous validation steps.
    # Store one dictionary per month; each dictionary becomes one output row.
    missing_columns = [
        flag_column
        for flag_column in streak_flags.values()
        if flag_column not in daily.columns
    ]

    if missing_columns:
        raise KeyError(
            "The following daily flag columns are missing: "
            + ", ".join(missing_columns)
        )

    # List of dictionaries, each dictionary constitute the results of a month
    rows = []

    for year_month, group in daily.groupby(
        "YEAR_MONTH",
        sort=True,
        observed=True
    ):
        row = {
            "YEAR_MONTH": year_month
        }

        for output_column, flag_column in streak_flags.items():
            row[output_column] = (
                longest_consecutive_event_days(
                    group=group,
                    flag_column=flag_column
                )
            )

        rows.append(row)

    return pd.DataFrame(
        rows,
        columns=[
            "YEAR_MONTH",
            *streak_flags.keys()
        ]
    )

Monthly continuous statistics must be calculated directly from the original cell-day observations.

This avoids taking an unweighted average of daily averages, which could distort the result if the number of available observations differed between days.

In [40]:
# Compute monthly statistics directly from all cell-day observations,
# including the average cell-level monthly precipitation accumulation.
def aggregate_raw_to_monthly(df):
    # These statistics are computed directly from all cell-day observations
    # without using the daily provincial aggregation as an intermediate step.
    monthly_raw = (
        df.groupby(
            "YEAR_MONTH",
            sort=True,
            observed=True
        )
        .agg(
            # Coverage
            OBSERVED_DAYS=(
                "DAY",
                "nunique"
            ),
            N_CELLS=(
                "IDCELL",
                "nunique"
            ),

            # Temperature: direct aggregation of all monthly cell-day values
            TEMPERATURE_AVG_MONTH=(
                "TEMPERATURE_AVG",
                "mean"
            ),
            TEMPERATURE_MAX_MONTH=(
                "TEMPERATURE_MAX",
                "max"
            ),
            TEMPERATURE_MIN_MONTH=(
                "TEMPERATURE_MIN",
                "min"
            ),
            
            # Precipitation
            PRECIPITATION_SUM_ALL_POINTS=(
                "PRECIPITATION",
                "sum"
            ),
            PRECIPITATION_DAILY_AVG=(
                "PRECIPITATION",
                "mean"
            ),
            PRECIPITATION_LOCAL_DAILY_MAX=(
                "PRECIPITATION",
                "max"
            ),

            # Wind
            WINDSPEED_AVG_MONTH=(
                "WINDSPEED",
                "mean"
            ),
            WINDSPEED_LOCAL_MAX=(
                "WINDSPEED",
                "max"
            ),

            # Other environmental variables
            VAPOURPRESSURE_AVG_MONTH=(
                "VAPOURPRESSURE",
                "mean"
            ),
            ET0_SUM_ALL_POINTS=(
                "ET0",
                "sum"
            ),
            ET0_DAILY_AVG=(
                "ET0",
                "mean"
            ),
            RADIATION_SUM_ALL_POINTS=(
                "RADIATION",
                "sum"
            ),
            RADIATION_DAILY_AVG=(
                "RADIATION",
                "mean"
            ),
        )
        .reset_index()
    )

    # These features represent monthly totals averaged spatially across grid cells.
    # For cumulative variables, summing every cell-day value would multiply
    # the provincial total by the number of cells. Dividing by N_CELLS gives
    # the average monthly accumulation across the equally sized cells.
    monthly_raw["PRECIPITATION_TOTAL_MONTH"] = (
        monthly_raw["PRECIPITATION_SUM_ALL_POINTS"]
        / monthly_raw["N_CELLS"]
    )
    monthly_raw["ET0_TOTAL_MONTH"] = (
        monthly_raw["ET0_SUM_ALL_POINTS"]
        / monthly_raw["N_CELLS"]
    )
    monthly_raw["RADIATION_TOTAL_MONTH"] = (
        monthly_raw["RADIATION_SUM_ALL_POINTS"]
        / monthly_raw["N_CELLS"]
    )

    monthly_raw = monthly_raw.drop(
        columns=[
            "PRECIPITATION_SUM_ALL_POINTS",
            "ET0_SUM_ALL_POINTS",
            "RADIATION_SUM_ALL_POINTS"
        ]
    )

    return monthly_raw

In [41]:
# Aggregate indicators whose definitions explicitly depend on the daily
# provincial state from the temporary daily dataframe.
# This does not introduce an average-of-averages error because these
# variables are defined only after the daily provincial aggregation.

def aggregate_daily_features_to_monthly(daily):
    
    monthly_daily = (
        daily.groupby(
            "YEAR_MONTH",
            sort=True,
            observed=True
        )
        .agg(
            # Used only to validate the merge with the raw monthly branch
            OBSERVED_DAYS_DAILY=(
                "DAY",
                "nunique"
            ),
            N_CELLS_DAILY=(
                "N_CELLS",
                "max"
            ),

            # Temporal variability of daily provincial temperature
            TEMPERATURE_DAILY_STD=(
                "TEMPERATURE_AVG_AREA",
                "std"
            ),

            # Mean spatial variability among cells across the days of the month
            TEMPERATURE_SPATIAL_STD_AVG=(
                "TEMPERATURE_SPATIAL_STD",
                "mean"
            ),

            # Daily precipitation features
            PRECIPITATION_AREA_DAILY_MAX=(
                "PRECIPITATION_AREA",
                "max"
            ),
            PRECIPITATION_SPATIAL_STD_AVG=(
                "PRECIPITATION_SPATIAL_STD",
                "mean"
            ),

            # Wet and dry days
            WET_DAYS_AREA_MEAN=(
                "WET_DAY_AREA_MEAN",
                "sum"
            ),
            WET_DAYS_SPATIAL=(
                "WET_DAY_SPATIAL",
                "sum"
            ),
            DRY_DAYS_AREA_MEAN=(
                "DRY_DAY_AREA_MEAN",
                "sum"
            ),
            DRY_DAYS_ALL_CELLS=(
                "DRY_DAY_ALL_CELLS",
                "sum"
            ),
            WET_AREA_FRACTION_AVG=(
                "WET_CELL_AREA_FRACTION",
                "mean"
            ),
            WET_AREA_FRACTION_MAX=(
                "WET_CELL_AREA_FRACTION",
                "max"
            ),
            WET_AREA_WHEN_PRESENT=(
                "WET_CELL_AREA_FRACTION",
                mean_when_positive
            ),

            # Absolute rain thresholds
            DAYS_RAIN_20=(
                "RAIN_20_DAY",
                "sum"
            ),
            DAYS_RAIN_50=(
                "RAIN_50_DAY",
                "sum"
            ),
            RAIN_20_AREA_FRACTION_MAX=(
                "RAIN_20_AREA_FRACTION",
                "max"
            ),
            RAIN_50_AREA_FRACTION_MAX=(
                "RAIN_50_AREA_FRACTION",
                "max"
            ),

            # Relative rain extremes
            DAYS_EXTREME_RAIN=(
                "EXTREME_RAIN_DAY",
                "sum"
            ),
            DAYS_VERY_EXTREME_RAIN=(
                "VERY_EXTREME_RAIN_DAY",
                "sum"
            ),
            EXTREME_RAIN_AREA_FRACTION_AVG=(
                "EXTREME_RAIN_AREA_FRACTION",
                "mean"
            ),
            EXTREME_RAIN_AREA_FRACTION_MAX=(
                "EXTREME_RAIN_AREA_FRACTION",
                "max"
            ),
            EXTREME_RAIN_AREA_WHEN_PRESENT=(
                "EXTREME_RAIN_AREA_FRACTION",
                mean_when_positive
            ),
            VERY_EXTREME_RAIN_AREA_FRACTION_MAX=(
                "VERY_EXTREME_RAIN_AREA_FRACTION",
                "max"
            ),

            # Heat
            DAYS_EXTREME_HEAT=(
                "EXTREME_HEAT_DAY",
                "sum"
            ),
            DAYS_VERY_EXTREME_HEAT=(
                "VERY_EXTREME_HEAT_DAY",
                "sum"
            ),
            EXTREME_HEAT_AREA_FRACTION_AVG=(
                "EXTREME_HEAT_AREA_FRACTION",
                "mean"
            ),
            EXTREME_HEAT_AREA_FRACTION_MAX=(
                "EXTREME_HEAT_AREA_FRACTION",
                "max"
            ),
            EXTREME_HEAT_AREA_WHEN_PRESENT=(
                "EXTREME_HEAT_AREA_FRACTION",
                mean_when_positive
            ),

            # Cold
            DAYS_EXTREME_COLD=(
                "EXTREME_COLD_DAY",
                "sum"
            ),
            EXTREME_COLD_AREA_FRACTION_AVG=(
                "EXTREME_COLD_AREA_FRACTION",
                "mean"
            ),
            EXTREME_COLD_AREA_FRACTION_MAX=(
                "EXTREME_COLD_AREA_FRACTION",
                "max"
            ),
            EXTREME_COLD_AREA_WHEN_PRESENT=(
                "EXTREME_COLD_AREA_FRACTION",
                mean_when_positive
            ),

            # Wind
            DAYS_EXTREME_WIND=(
                "EXTREME_WIND_DAY",
                "sum"
            ),
            EXTREME_WIND_AREA_FRACTION_AVG=(
                "EXTREME_WIND_AREA_FRACTION",
                "mean"
            ),
            EXTREME_WIND_AREA_FRACTION_MAX=(
                "EXTREME_WIND_AREA_FRACTION",
                "max"
            ),
            EXTREME_WIND_AREA_WHEN_PRESENT=(
                "EXTREME_WIND_AREA_FRACTION",
                mean_when_positive
            ),

            # Absolute temperature thresholds
            DAYS_FROST_ANY_AREA=(
                "FROST_AREA_FRACTION",
                lambda values: int((values > 0).sum())
            ),
            # It is possible to write it in this way thanks to the already existing boolean column
            # Counting True values is equivalent to summing a boolean column.
            # The same logic applies to DAYS_HOT_30_10PCT and DAYS_HOT_35_10PCT.
            DAYS_FROST_10PCT=(
                "FROST_DAY_10PCT",
                "sum" 
            ),
            DAYS_HOT_30_ANY_AREA=(
                "HOT_30_AREA_FRACTION",
                lambda values: int((values > 0).sum())
            ),
            DAYS_HOT_30_10PCT=(
                "HOT_30_DAY_10PCT",
                "sum"
            ),
            DAYS_HOT_35_ANY_AREA=(
                "HOT_35_AREA_FRACTION",
                lambda values: int((values > 0).sum())
            ),
            DAYS_HOT_35_10PCT=(
                "HOT_35_DAY_10PCT",
                "sum"
            ),
        )
        .reset_index()
    )

    monthly_streaks = aggregate_streaks_to_monthly(
        daily=daily,
        streak_flags=STREAK_FLAGS
    )

    monthly_daily = monthly_daily.merge(
        monthly_streaks,
        on="YEAR_MONTH",
        how="left",
        validate="one_to_one"
    )

    return monthly_daily

Before assembling the final dataset, the static altitude information must be summarized at provincial level.

Altitude is associated with grid cells rather than individual dates, so each unique cell must contribute only once to the provincial altitude statistics.

In [42]:
# Altitude is a static property of each grid cell.
# It must therefore be aggregated once per unique cell,
# without counting the same cell again for every observed day.

def calculate_province_altitude_features(df):

    altitude_by_cell = (
        df.groupby(
            "IDCELL",
            observed=True
        )["ALTITUDE"]
        .agg(
            altitude_values="nunique",
            altitude="first"
        )
    )

    # Each cell must have one and only one altitude value
    inconsistent_cells = (
        altitude_by_cell["altitude_values"] != 1
    )

    if inconsistent_cells.any():
        problematic_cells = (
            altitude_by_cell
            .loc[inconsistent_cells]
            .index
            .tolist()
        )

        raise ValueError(
            "Some cells have inconsistent altitude values: "
            + ", ".join(map(str, problematic_cells))
        )

    altitude = altitude_by_cell["altitude"]

    return {
        "ALTITUDE_MEAN": altitude.mean(),
        "ALTITUDE_STD": altitude.std(),
        "ALTITUDE_MIN": altitude.min(),
        "ALTITUDE_MAX": altitude.max(),
        "ALTITUDE_RANGE": (
            altitude.max() - altitude.min()
        )
    }

Finally, it is possible to bring the different elements together and build the final dataframe for the current province.

In [43]:
def build_monthly_dataset(df, daily):
    monthly_raw = aggregate_raw_to_monthly(df)
    monthly_daily = aggregate_daily_features_to_monthly(daily)

    monthly = monthly_raw.merge(
        monthly_daily,
        on="YEAR_MONTH",
        how="inner",
        validate="one_to_one"
    )

    # The two branches must describe the same monthly coverage.
    inconsistent_coverage = (
        (monthly["OBSERVED_DAYS"] != monthly["OBSERVED_DAYS_DAILY"])
        | (monthly["N_CELLS"] != monthly["N_CELLS_DAILY"])
    )

    if inconsistent_coverage.any():
        problematic_months = monthly.loc[
            inconsistent_coverage,
            ["YEAR_MONTH"]
        ]
        raise ValueError(
            "Raw and daily aggregation branches have inconsistent coverage: "
            + problematic_months.to_string(index=False)
        )

    monthly = monthly.drop(
        columns=[
            "OBSERVED_DAYS_DAILY",
            "N_CELLS_DAILY"
        ]
    )

    # Add static provincial altitude features
    altitude_features = (
        calculate_province_altitude_features(df)
    )

    monthly = monthly.assign(
        **altitude_features
    )

    return monthly

In [44]:
monthly = build_monthly_dataset(
    df=df,
    daily=daily
)

In [45]:
display(monthly.head())

,YEAR_MONTH,OBSERVED_DAYS,N_CELLS,TEMPERATURE_AVG_MONTH,TEMPERATURE_MAX_MONTH,TEMPERATURE_MIN_MONTH,PRECIPITATION_DAILY_AVG,PRECIPITATION_LOCAL_DAILY_MAX,WINDSPEED_AVG_MONTH,WINDSPEED_LOCAL_MAX,...,MAX_CONSECUTIVE_DRY_DAYS_AREA_MEAN,MAX_CONSECUTIVE_WET_DAYS_AREA_MEAN,MAX_CONSECUTIVE_FROST_10PCT_DAYS,MAX_CONSECUTIVE_HOT_30_10PCT_DAYS,MAX_CONSECUTIVE_HOT_35_10PCT_DAYS,ALTITUDE_MEAN,ALTITUDE_STD,ALTITUDE_MIN,ALTITUDE_MAX,ALTITUDE_RANGE
0,1980-01,31,56,0.701843,14.6,-11.6,2.475806,31,0.631567,7.1,...,8,4,31,0,0,243.017857,189.006829,77,932,855
1,1980-02,29,56,4.458805,16.7,-6.5,0.087438,8,0.622044,5.4,...,16,1,13,0,0,243.017857,189.006829,77,932,855
2,1980-03,31,56,7.090207,19.3,-4.2,3.455069,35,1.102995,6.3,...,5,5,3,0,0,243.017857,189.006829,77,932,855
3,1980-04,30,56,10.264821,22.4,-2.9,0.380952,9,1.730238,6.7,...,14,2,2,0,0,243.017857,189.006829,77,932,855
4,1980-05,31,56,13.724885,23.9,2.2,3.220046,27,1.081682,5.9,...,3,6,0,0,0,243.017857,189.006829,77,932,855


In [46]:
print('The table has',monthly.shape[0],'rows and',monthly.shape[1],'columns.')

The table has 552 rows and 72 columns.


## Example output

For demonstration purposes, the monthly dataset is labelled with the province name derived from the selected source object and exported as an uncompressed CSV file in the local project `data/` directory.

In the production pipeline, the monthly results from all mapped provinces are collected, concatenated, enriched with the official province and region labels, and exported as the complete climate dataset.

In [47]:
monthly["PROVINCE"] = FILENAME.split(".")[0]

In [48]:
# Save the single-province test output locally as an uncompressed CSV
output_path = OUTPUT_DIR / ("TEST_" + FILENAME.replace(".gz",""))

monthly.to_csv(
    output_path,
    index=False
)